In [1]:
import os
path = os.getcwd()
path

'D:\\Project 2\\trial2\\data'

In [2]:
folders = [x[0] for x in os.walk(path + '\\data')]
folders = folders[1:]
folders

['D:\\Project 2\\trial2\\data\\data\\고심이',
 'D:\\Project 2\\trial2\\data\\data\\늬에시',
 'D:\\Project 2\\trial2\\data\\data\\루피',
 'D:\\Project 2\\trial2\\data\\data\\오니기리',
 'D:\\Project 2\\trial2\\data\\data\\이상한 애']

In [3]:
import numpy as np 
import pandas as pd 
from PIL import Image 
import matplotlib.pyplot as plt 
import glob 
import cv2 as cv 

#데이터 증식
step = 2 
kernelSize = 3 
size = 100 
angleRange = 90 
angleRange //= 2

#categories = [i.split('\\')[5] for i in folders] 
#categories = pd.DataFrame(zip(list(range(len(categories))),categories), columns=['ID,Category'.split(',')]) 
#categories.to_csv('./trial.csv',encoding='cp949',index=False)

categories = pd.read_csv('./trial.csv',encoding='cp949')

categories = list(categories.Category)
i = 0
for category in categories:
    newPath = path + '\\newData\\%d' % (i)
    i += 1
    if not os.path.isdir(newPath):
        os.mkdir(newPath)
    relPath = str(path + "\\data\\" + category)
    files = glob.glob(relPath + "\\*.jpg")
    
    removeFiles = glob.glob(newPath + "\\*.png")
    for f in removeFiles:
        os.remove(f)
    
    #continue
    
    j = 0
    for f in files:
        
        img = Image.open(f)
        img = img.convert('L')
        img = np.asarray(img)
        img = cv.resize(src=img, dsize=(size,size))
        blurredImg = cv.GaussianBlur(img, (kernelSize, kernelSize),0)
        processed = cv.Canny(blurredImg, 30, 150)
        
        cv.imwrite(newPath + "\\%d_resized.png" % (j), processed)
        
        
        center = (size//2,size//2)
        scale = 1.0
        
        for k in range(-angleRange,angleRange+1,step):
            if k == 0:
                continue
            mtx = cv.getRotationMatrix2D(center, k, scale)
            img2 = cv.warpAffine(processed, mtx, (size,size))
            cv.imwrite(newPath + "\\%d_resized_%d.png" % (j,k), img2)    
            
        j += 1

C:\Users\user\.conda\envs\dl_study\lib\site-packages\PIL\Image.py:974: UserWarning: Palette images with Transparency expressed in bytes should be converted to RGBA images
  "Palette images with Transparency expressed in bytes should be "


In [4]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import glob
import cv2 as cv
import os

## 데이터 증식

#step = 1
maxData = 300
categories = pd.read_csv('trial.csv',encoding='cp949')
categories2 = (categories.Category)

xFiles = []
y = []
for i in range(len(categories2)):
    
    newPath = path + '\\newData\\%d' % (i)
    
    files = glob.glob(newPath + '\*.png')
    #files = [j for j in files if len(j.split('_')) != 1]
    #files = files[::2]
    xFiles.extend(files)
    y.extend([categories2[i]]*len(files))
    
y = pd.Series(y)
y = pd.get_dummies(y,prefix='y',prefix_sep="_").astype('int32')

#y = [1 if i == 'positive' else 0 for i in y]

y = np.array(y).astype('int32')
X = []
i = 0
for f in xFiles:
    
    img = cv.imread(f, cv.IMREAD_GRAYSCALE)
    #img = cv.cvtColor(img, cv.COLOR_BGR2GRAY)
    #img = img.resize((49,49))
    img = np.asarray(img)
    img = img.astype('float32') / 255
    #img = cv.cvtColor(img, cv.COLOR_BGR2GRAY)
     #img = img.convert('RGBA')
    #img = img.convert("A")
    #img = cv.cvtColor(img,cv.COLOR_BGR2GRAY)
    X.append(img)

#import tensorflow.keras
import numpy as np
#from tensorflow.keras.utils import to_categorical

# 입력과 출력 지정하기 
nb_classes = len(categories2) # 레이블 수
X = np.array(X)
X = X.reshape(-1, size,size,1)

In [5]:
y.shape

(8225, 5)

In [6]:
categories

,ID,Category
0,0,고심이
1,1,늬에시
2,2,루피
3,3,오니기리
4,4,이상한 애


In [7]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(X,y,
                                                   test_size=0.33,
                                                   random_state=777,
                                                   stratify=y)

X_train, X_val, y_train, y_val = train_test_split(X_train, y_train,
                                                 test_size=0.2,
                                                 random_state=777,
                                                 stratify=y_train)

In [8]:
X.shape

(8225, 100, 100, 1)

In [9]:
import tensorflow.keras
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout, Flatten
from tensorflow.keras.layers import Conv2D, MaxPooling2D
from tensorflow.keras.optimizers import Adam

# CNN 모델 정의하기
model = Sequential([
    Conv2D(32, kernel_size=(3,3), activation='relu',input_shape=(size,size, 1)),
    Conv2D(64, kernel_size=(3,3), activation='relu'),
    MaxPooling2D(pool_size=(2,2)),
    Dropout(0.25),
        
    Conv2D(64, kernel_size=(3,3), activation='relu'),
    Conv2D(64, kernel_size=(3,3), activation='relu'),
    MaxPooling2D(pool_size=(2,2)),
    Dropout(0.25),
    
        
    Flatten(),
    Dense(512, activation='relu'),
    Dropout(0.5),
    Dense(nb_classes,activation='softmax')
        
])

In [10]:
model.summary()

Model: "sequential"
_________________________________________________________________
Layer (type)                 Output Shape              Param #   
conv2d (Conv2D)              (None, 98, 98, 32)        320       
_________________________________________________________________
conv2d_1 (Conv2D)            (None, 96, 96, 64)        18496     
_________________________________________________________________
max_pooling2d (MaxPooling2D) (None, 48, 48, 64)        0         
_________________________________________________________________
dropout (Dropout)            (None, 48, 48, 64)        0         
_________________________________________________________________
conv2d_2 (Conv2D)            (None, 46, 46, 64)        36928     
_________________________________________________________________
conv2d_3 (Conv2D)            (None, 44, 44, 64)        36928     
_________________________________________________________________
max_pooling2d_1 (MaxPooling2 (None, 22, 22, 64)        0

In [11]:
model.compile(loss='categorical_crossentropy',
             optimizer=Adam(1e-4),
             metrics=['accuracy'])

In [ ]:
hist = model.fit(X_train,y_train, 
         batch_size=64,
         epochs=15,
         verbose=1,
         validation_data=(X_val, y_val))

Epoch 1/15
69/69 [==============================] - 132s 2s/step - loss: 1.0111 - accuracy: 0.6039 - val_loss: 0.5829 - val_accuracy: 0.8167
Epoch 2/15
69/69 [==============================] - 132s 2s/step - loss: 0.4567 - accuracy: 0.8305 - val_loss: 0.2675 - val_accuracy: 0.9446
Epoch 3/15
69/69 [==============================] - 133s 2s/step - loss: 0.2052 - accuracy: 0.9299 - val_loss: 0.1240 - val_accuracy: 0.9710
Epoch 4/15
69/69 [==============================] - 135s 2s/step - loss: 0.0843 - accuracy: 0.9771 - val_loss: 0.0657 - val_accuracy: 0.9809
Epoch 5/15
69/69 [==============================] - 141s 2s/step - loss: 0.0325 - accuracy: 0.9941 - val_loss: 0.0426 - val_accuracy: 0.9846
Epoch 6/15
 4/69 [>.............................] - ETA: 2:07 - loss: 0.0123 - accuracy: 1.0000

In [ ]:
# 모델 평가하기 
score = model.evaluate(X_test, y_test, verbose=1)
print('정답률=', score[1], 'loss=', score[0])

In [ ]:
modelJSON = model.to_json()
with open (path+"modelCategory.json", "w") as json_file:
    json_file.write(modelJSON)
model.save_weights(path+"\\modelCategory.h5")

jsonFile = open(path+'modelCategory.json','r')
loadedModel = jsonFile.read()
jsonFile.close()

from keras.models import model_from_json
loadedModel = model_from_json(loadedModel)
loadedModel.load_weights("modelCategory.h5")

loadedModel.compile(loss='categorical_crossentropy', optimizer=Adam(1e-4), metrics=['accuracy'])
score = loadedModel.evaluate(X_test, y_test, verbose=1)

In [ ]:
from PIL import Image

image = Image.open(path +'\\trial3.png')
image = image.convert('L')
image = image.resize((size,size))
image.save(path + 'dummy.png')
image = np.array(image)
image = ~image
plt.imshow(image)
imarray = np.array(image).astype('float32') / 255

In [ ]:
X_test2 = np.array(imarray)
X_test2 = X_test2.reshape(-1,size,size,1)

y_test2 = model.predict(X_test2)
index = np.argmax(y_test2)
categories2[index]

In [ ]:
# 학습 상태를 그래프로 그리기 
import matplotlib.pyplot as plt

his_dict = hist.history
loss = his_dict['loss']
val_loss = his_dict['val_loss'] 

epochs = range(1, len(loss) + 1)
fig = plt.figure(figsize = (10, 5))

# 학습 및 데트스 손실(loss) 그리기
ax1 = fig.add_subplot(1, 2, 1)
ax1.plot(epochs, loss, color = 'blue', label = 'train_loss')
ax1.plot(epochs, val_loss, color = 'orange', label = 'val_loss')
ax1.set_title('train and val loss')
ax1.set_xlabel('epochs')
ax1.set_ylabel('loss')
ax1.legend()

acc = his_dict['accuracy']
val_acc = his_dict['val_accuracy']

# 학습 및 데이타 정확도(accuracy) 그리기
ax2 = fig.add_subplot(1, 2, 2)
ax2.plot(epochs, acc, color = 'blue', label = 'train_acc')
ax2.plot(epochs, val_acc, color = 'orange', label = 'val_acc')
ax2.set_title('train and val acc')
ax2.set_xlabel('epochs')
ax2.set_ylabel('acc')
ax2.legend()

plt.show()
fig.savefig(path + 'modelCategory.png')